# Imports

In [24]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch_geometric
from torch_geometric.data import Data
from scipy.spatial.distance import cdist
import pickle
import rasterio
import glob
import tifffile
import matplotlib.pyplot as plt
import re
import random

# ______________________

In [12]:
dataset_fp = "/users/bumjin/Documents/Brown Notes/Singh Lab/QBAM/Data/amd/donor"
# donor_df_fp = "AMD_Donor-Match_DNNI.csv"
# donor_df = pd.read_csv(f"{dataset_fp}/{donor_df_fp}")

In [47]:
donor1_list = np.array([os.path.basename(fp) for fp in list(glob.glob(f"{dataset_fp}/csvs/1*"))])
donor2_list = np.array([os.path.basename(fp) for fp in list(glob.glob(f"{dataset_fp}/csvs/2*"))])
donor3_list = np.array([os.path.basename(fp) for fp in list(glob.glob(f"{dataset_fp}/csvs/3*"))])

# Imports

In [48]:
print(len(donor1_list), len(donor2_list), len(donor3_list))

522 64 444


In [49]:
donor1_list[:5]

array(['1B_D74_n599.ome.tif.csv', '1B_D75_n705.ome.tif.csv',
       '1A_D75_n1027.ome.tif.csv', '1B_D75_n667.ome.tif.csv',
       '1B_D77_n967.ome.tif.csv'], dtype='<U24')

In [50]:
donor1_pos = np.vstack((donor1_list, donor1_list)).T
np.random.shuffle(donor1_pos)

donor2_pos = np.vstack((donor2_list, donor2_list)).T
np.random.shuffle(donor2_pos)

donor3_pos = np.vstack((donor3_list, donor3_list)).T
np.random.shuffle(donor3_pos)

In [55]:
donor1_neg = np.vstack(
    (np.random.choice(donor2_list, size=len(donor1_list), replace=True),
     np.random.choice(donor3_list, size=len(donor1_list), replace=True))
     ).T

donor2_neg = np.vstack(
    (np.random.choice(donor1_list, size=len(donor2_list), replace=True),
     np.random.choice(donor3_list, size=len(donor2_list), replace=True))
     ).T

donor3_neg = np.vstack(
    (np.random.choice(donor1_list, size=len(donor3_list), replace=True),
     np.random.choice(donor2_list, size=len(donor3_list), replace=True))
     ).T

In [56]:
print(donor1_neg.shape, donor2_neg.shape, donor3_neg.shape, )

(522, 2) (64, 2) (444, 2)


In [62]:
all_donor1 = np.hstack((donor1_list[:, None], donor1_pos, donor1_neg))
all_donor2 = np.hstack((donor2_list[:, None], donor2_pos, donor2_neg))
all_donor3 = np.hstack((donor3_list[:, None], donor3_pos, donor3_neg))

all_sample = np.vstack((all_donor1, all_donor2, all_donor3))


In [63]:
all_sample.shape

(1030, 5)

In [66]:
donor_df = pd.DataFrame({'Sample': all_sample[:, 0], 'Positive 1': all_sample[:, 1], 'Positive 2': all_sample[:, 2], 'Negative 1': all_sample[:, 3], 'Negative 2': all_sample[:, 4]})
donor_df

,Sample,Positive 1,Positive 2,Negative 1,Negative 2
0,1B_D74_n599.ome.tif.csv,1B_D76_n896.ome.tif.csv,1B_D76_n896.ome.tif.csv,2B_D73_n125.ome.tif.csv,3B_D73_n290.ome.tif.csv
1,1B_D75_n705.ome.tif.csv,1A_D75_n1086.ome.tif.csv,1A_D75_n1086.ome.tif.csv,2B_D73_n120.ome.tif.csv,3B_D74_n498.ome.tif.csv
2,1A_D75_n1027.ome.tif.csv,1B_D76_n790.ome.tif.csv,1B_D76_n790.ome.tif.csv,2B_D73_n122.ome.tif.csv,3B_D74_n497.ome.tif.csv
3,1B_D75_n667.ome.tif.csv,1B_D76_n824.ome.tif.csv,1B_D76_n824.ome.tif.csv,2B_D73_n91.ome.tif.csv,3B_D73_n348.ome.tif.csv
4,1B_D77_n967.ome.tif.csv,1A_D75_n1089.ome.tif.csv,1A_D75_n1089.ome.tif.csv,2B_D73_n79.ome.tif.csv,3B_D74_n471.ome.tif.csv
...,...,...,...,...,...
1025,3B_D73_n376.ome.tif.csv,3B_D74_n520.ome.tif.csv,3B_D74_n520.ome.tif.csv,1B_D77_n939.ome.tif.csv,2B_D73_n84.ome.tif.csv
1026,3C_D75_n166.ome.tif.csv,3C_D75_n178.ome.tif.csv,3C_D75_n178.ome.tif.csv,1A_D75_n1072.ome.tif.csv,2B_D73_n99.ome.tif.csv
1027,3B_D73_n328.ome.tif.csv,3B_D73_n382.ome.tif.csv,3B_D73_n382.ome.tif.csv,1B_D74_n641.ome.tif.csv,2B_D73_n103.ome.tif.csv
1028,3B_D73_n358.ome.tif.csv,3C_D75_n245.ome.tif.csv,3C_D75_n245.ome.tif.csv,1B_D76_n878.ome.tif.csv,2B_D73_n75.ome.tif.csv


In [102]:
def get_features(centroid_data):

    features = torch.tensor(centroid_data[['Area', 'Aspect_Ratio_BB', 'BB_Xmin', 'BB_Ymin', 'BB_Width', 'BB_Height', 
                              'Center_BB_X', 'Center_BB_Y', 'Centroid_X', 'Centroid_Y', 'Circularity', 
                              'Distance_From_Border', 'Eccentricity', 'Entropy', 'ExtendBB', 'Mean', 
                              'Median', 'Mode', 'Orientation', 'Perimeter', 'StandardDeviation', 
                              'Skewness', 'Kurtosis', 'Hyperskewness', 'Hyperflatness', 'TContrast_Average', 
                              'TContrast_Ortho_principal_component_value', 'TContrast_Principal_component_angle', 
                              'TContrast_Principal_component_value', 'TCorrelation_Average', 
                              'TCorrelation_Ortho_principal_component_value', 'TCorrelation_Principal_component_angle', 
                              'TCorrelation_Principal_component_value', 'THomogeneity_Average', 
                              'THomogeneity_Ortho_principal_component_value', 'THomogeneity_Principal_component_angle', 
                              'THomogeneity_Principal_component_value', 'TEnergy_Average', 
                              'TEnergy_Ortho_principal_component_value', 'TEnergy_Principal_component_angle', 
                              'TEnergy_Principal_component_value', 'TVariance_Average', 
                              'TVariance_Ortho_principal_component_value', 'TVariance_Principal_component_angle', 
                              'TVariance_Principal_component_value', 'TEntropy_Average', 
                              'TEntropy_Ortho_principal_component_value', 'TEntropy_Principal_component_angle', 
                              'TEntropy_Principal_component_value', 'TInvDiffMoment_Average', 
                              'TInvDiffMoment_Ortho_principal_component_value', 'TInvDiffMoment_Principal_component_angle', 
                              'TInvDiffMoment_Principal_component_value', 'TSumAverage_Average', 
                              'TSumAverage_Ortho_principal_component_value', 'TSumAverage_Principal_component_angle', 
                              'TSumAverage_Principal_component_value', 'TSumVariance_Average', 
                              'TSumVariance_Ortho_principal_component_value', 'TSumVariance_Principal_component_angle', 
                              'TSumVariance_Principal_component_value', 'TSumEntropy_Average', 
                              'TSumEntropy_Ortho_principal_component_value', 'TSumEntropy_Principal_component_angle', 
                              'TSumEntropy_Principal_component_value', 'TDiffAverage_Average', 
                              'TDiffAverage_Ortho_principal_component_value', 'TDiffAverage_Principal_component_angle', 
                              'TDiffAverage_Principal_component_value', 'TDiffVariance_Average', 
                              'TDiffVariance_Ortho_principal_component_value', 'TDiffVariance_Principal_component_angle', 
                              'TDiffVariance_Principal_component_value', 'TDiffEntropy_Average', 
                              'TDiffEntropy_Ortho_principal_component_value', 'TDiffEntropy_Principal_component_angle', 
                              'TDiffEntropy_Principal_component_value'

                              ]].values.astype(np.float32))

    return features

In [92]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

class PairData(Data):
    def __inc__(self, key, value, *args, **kwargs):
        if key == 'edge_index1':
            return self.x1.size(0)
        if key == 'edge_index2':
            return self.x2.size(0)
        if key == 'edge_weights1':
            return self.x1.size(0)
        if key == 'edge_weights2':
            return self.x2.size(0)
        return super().__inc__(key, value, *args, **kwargs)

In [103]:
def get_graph_pair(centroids_fp1, centroids_fp2):
    num_neighbours = 20
    def get_graph(centroids_fp):
        centroids = pd.read_csv(centroids_fp)

        # Drop unnecessary column
        if 'Partition_ID' in centroids.columns:
            centroids = centroids.drop('Partition_ID', axis=1)

        features = get_features(centroids)

        # Create a graph
        sp_centroids = centroids[['Centroid_X', 'Centroid_Y']].values
        num_nodes = len(sp_centroids)
        euc_distances = cdist(sp_centroids, sp_centroids)
        closest_neighbours = np.argsort(euc_distances, axis=1)[:, :num_neighbours + 1]
        edge_array = np.array([np.repeat(range(num_nodes), num_neighbours + 1), np.reshape(closest_neighbours, (-1))])
        edge_index = torch.tensor(edge_array, dtype=torch.long)

        # Calculate edge weights based on distances
        closest_distances = np.sort(euc_distances, axis=1)[:, :num_neighbours + 1]
        edge_weights = np.exp(-np.reshape(closest_distances, (-1)))
        edge_weights = torch.tensor(edge_weights, dtype=torch.float32)

        edge_index, edge_weights = torch_geometric.utils.to_undirected(edge_index, edge_weights)

        return features, edge_index, edge_weights
    
    features1, edge_index1, edge_weights1 = get_graph(centroids_fp1)
    features2, edge_index2, edge_weights2 = get_graph(centroids_fp2)

    data = PairData(x1=features1, edge_index1=edge_index1, edge_weights1=edge_weights1,
                    x2=features2, edge_index2=edge_index2, edge_weights2=edge_weights2)

    return data

In [ ]:
output_filename = "pairwise_donors.pkl"
data_list = []

for _, row in donor_df.iterrows():
    sample = f"{dataset_fp}/csvs/{row["Sample"]}"
    pos1 = f"{dataset_fp}/csvs/{row["Positive 1"]}"
    pos2 = f"{dataset_fp}/csvs/{row["Positive 2"]}"
    neg1 = f"{dataset_fp}/csvs/{row["Negative 1"]}"
    neg2 = f"{dataset_fp}/csvs/{row["Negative 2"]}"

    pos_pair1 = get_graph_pair(sample, pos1)
    pos_pair1.y = torch.tensor([1.0], dtype=torch.float)
    pos_pair2 = get_graph_pair(sample, pos2)
    pos_pair2.y = torch.tensor([1.0], dtype=torch.float)
    neg_pair1 = get_graph_pair(sample, neg1)
    neg_pair1.y = torch.tensor([0.0], dtype=torch.float)
    neg_pair2 = get_graph_pair(sample, neg2)
    neg_pair2.y = torch.tensor([0.0], dtype=torch.float)

    data_list += [pos_pair1, pos_pair2, neg_pair1, neg_pair2]

random.shuffle(data_list)

train = data_list[:int(.8 * len(data_list))]
val = data_list[int(.8 * len(data_list)):int(.9 * len(data_list))]
test = data_list[int(.9 * len(data_list)):]

for split, name in zip((train,val,test), ("train", "val", "test")):
    with open(f"{dataset_fp}/{name}_{output_filename}", 'wb') as f:
        pickle.dump(train, f)

print("Feature extraction and graph construction completed. Data saved to:", output_filename)


Feature extraction and graph construction completed. Data saved to: pairwise_donors.pkl


In [125]:
train[0].edge_index1.shape

torch.Size([2, 3123])

In [128]:
loader = DataLoader(train, batch_size=16, follow_batch=["x1", "x2"])
batch = next(iter(loader))
batch

PairDataBatch(x1=[2457, 77], x1_batch=[2457], x1_ptr=[17], edge_index1=[2, 58453], edge_weights1=[58453], x2=[2628, 77], x2_batch=[2628], x2_ptr=[17], edge_index2=[2, 62390], edge_weights2=[62390], y=[16])

In [120]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, Identity
from collections import namedtuple

DataPoint = namedtuple('DataPoint', ['x', 'edge_index', 'batch'])

class GCN_Merge(torch.nn.Module):
    def __init__(self, gnn1, gnn2):
        super().__init__()
        self.gnn1 = gnn1
        self.gnn2 = gnn2

        # for param in gnn1.parameters():
        #     param.requires_grad = False
        # for param in gnn2.parameters():
        #     param.requires_grad = False

        self.gnn1.out_linear = Identity()
        self.gnn2.out_linear = Identity()

        self.dropout = Dropout(p=0.5)

        self.pred_head = Linear(gnn1.dense_hidden * 2, 1)
        for param in self.pred_head.parameters():
            param.requires_grad = False

    def forward(self, pair_data):
        data_1, data_2 = DataPoint(pair_data.x1, pair_data.edge_index1, pair_data.x1_batch), \
                         DataPoint(pair_data.x2, pair_data.edge_index2, pair_data.x2_batch)
        
        emb1, emb2 = self.gnn1(data_1), self.gnn2(data_2)

        x = torch.cat((emb1, emb2), dim=-1)
        x = self.dropout(x)
        x = self.pred_head(x)
        x = F.sigmoid(x)

        return x

In [121]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, BatchNorm, JumpingKnowledge
from torch.nn import Linear, Dropout

class GCN_G4_D5(torch.nn.Module):
    def __init__(self, num_node_features, output_dim, hidden_channels=128, dense_hidden=128, num_heads=8, dropout_p=0.5):
        super().__init__()
        self.num_node_features = num_node_features
        self.output_dim = output_dim
        self.hidden_channels = hidden_channels
        self.dense_hidden = dense_hidden
        self.num_heads = num_heads
        self.edge_dim = 0 if output_dim == 2 else 1

        self.conv1 = GATv2Conv(self.num_node_features, self.hidden_channels, heads=self.num_heads, concat=True, edge_dim=self.edge_dim)
        self.norm1 = BatchNorm(self.hidden_channels * self.num_heads)

        self.conv2 = GATv2Conv(self.hidden_channels * self.num_heads, self.hidden_channels, heads=self.num_heads, concat=True, edge_dim=self.edge_dim)
        self.norm2 = BatchNorm(self.hidden_channels * self.num_heads)

        self.conv3 = GATv2Conv(self.hidden_channels * self.num_heads, self.hidden_channels, heads=self.num_heads, concat=True, edge_dim=self.edge_dim)
        self.norm3 = BatchNorm(self.hidden_channels * self.num_heads)
        
        self.conv4 = GATv2Conv(self.hidden_channels * self.num_heads, self.output_dim, heads=1, concat=False, edge_dim=self.edge_dim)
        self.norm4 = BatchNorm(self.output_dim)
        
        self.dropout = Dropout(p=dropout_p)

        self.linear1 = Linear(self.output_dim, self.dense_hidden)
        self.linear2 = Linear(self.dense_hidden, self.dense_hidden)
        self.linear3 = Linear(self.dense_hidden, self.dense_hidden)
        self.linear4 = Linear(self.dense_hidden, self.dense_hidden)
        self.out_linear = Linear(self.dense_hidden, self.output_dim)


    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.dropout(x)
        x = self.norm1(x)
        x = F.leaky_relu(x)

        x = self.conv2(x, edge_index)
        x = self.dropout(x)
        x = self.norm2(x)
        x = F.leaky_relu(x)

        x = self.conv3(x, edge_index)
        x = self.dropout(x)
        x = self.norm3(x)
        x = F.leaky_relu(x)

        x = self.conv4(x, edge_index)
        x = self.dropout(x)
        x = self.norm4(x)
        x = F.leaky_relu(x)

        x = global_mean_pool(x, batch)

        x = F.leaky_relu(self.linear1(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.linear2(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.linear3(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.linear4(x))
        x = self.dropout(x)
        x = self.out_linear(x)

        return x

In [122]:
hidden_size = 144
dense_hidden = 512
learning_rate = 0.001
lr_decay = 0.5
weight_decay = 0.005
dropout_rate = 0.5

model1 = GCN_G4_D5(77, 1, hidden_channels = hidden_size, dense_hidden = dense_hidden, dropout_p=dropout_rate)
model2 = GCN_G4_D5(77, 1, hidden_channels = hidden_size, dense_hidden = dense_hidden, dropout_p=dropout_rate)
gnn_merge = GCN_Merge(model1, model2)

In [123]:
gnn_merge(batch).shape

torch.Size([16, 1])